[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/07_web_crawling/07_web_crawling.ipynb)

# 07. 웹 크롤링 실습 (requests + BeautifulSoup + DB 저장)

> 관련 예제 프로젝트: [`example-projects/crawl-storage-example`](../../example-projects/crawl-storage-example) (A파트-1: 수집/저장)

이 노트북은 example-projects의 크롤링 예제에서 쓰인 라이브러리(`requests`, `beautifulsoup4`, `python-dotenv`, `psycopg2`)를 하나씩 직접 실습해보는 튜토리얼입니다.

## 이론

### 1) 왜 크롤링한 원본을 바로 검색엔진에 넣지 않을까?
크롤링은 네트워크 요청이 많아 느리고 실패할 수 있습니다. 원본(raw)을 먼저 DB에 그대로 저장해두면, 나중에 청킹 방식을 바꾸더라도 크롤링을 다시 할 필요가 없습니다. 원본과 가공본을 분리해두면 문제가 생겼을 때 원인 추적도 쉬워집니다.

### 2) 전체 흐름
```
URL 목록 -> requests로 페이지/파일 요청
         -> BeautifulSoup으로 HTML 파싱 (본문 텍스트 추출)
         -> (PDF 링크면) 바이너리 그대로 저장
         -> DB에 원본 저장
```

### 3) 실습 대상 사이트와 크롤링 예의
실제 서비스 사이트를 무분별하게 긁으면 안 되므로, 이 노트북에서는 크롤링 연습을 위해 공개된 연습용 사이트인 [books.toscrape.com](https://books.toscrape.com/)과 [quotes.toscrape.com](https://quotes.toscrape.com/)을 사용합니다. 두 사이트 모두 스크래핑 연습 용도로 만들어진 정적 사이트라 안심하고 반복 요청할 수 있습니다.

실전에서 다른 사이트를 크롤링할 때는 먼저 `https://대상사이트/robots.txt`를 확인해서 크롤링이 금지된 경로는 없는지, 요청 빈도 제한(`Crawl-delay`)이 있는지 살펴보는 것이 예의이자 관례입니다. `crawl.py`가 요청 사이에 `CRAWL_DELAY_SECONDS`만큼 쉬어가는 것도 같은 맥락입니다.

### 4) DB는 PostgreSQL 대신 sqlite3로 연습
원본 프로젝트는 `psycopg2`로 PostgreSQL에 저장하지만, Colab/로컬 노트북에서는 별도 DB 서버를 띄우기 번거롭습니다. 그래서 이 노트북에서는 파이썬 표준 라이브러리인 `sqlite3`로 **거의 동일한 SQL 패턴** (테이블 생성, UPSERT)을 연습합니다. `ON CONFLICT ... DO UPDATE`(UPSERT) 문법은 PostgreSQL 9.5+와 SQLite 3.24+가 공통으로 지원하는 확장 문법이라(ANSI SQL 표준 자체는 아니지만 두 엔진에서 동일하게 동작합니다), 실제 PostgreSQL이 있다면 `sqlite3.connect(...)`를 `psycopg2.connect(DATABASE_URL)`로 바꾸는 것만으로 그대로 옮겨갈 수 있습니다.

## 실습 0. 환경 설정

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
print("Running in Colab:", IN_COLAB)

if IN_COLAB:
    !pip install -q requests beautifulsoup4 python-dotenv

## 실습 1. `requests`로 페이지 가져오기

`crawl.py`의 `fetch()` 함수와 같은 패턴입니다: 브라우저인 척하는 헤더를 붙이고, `timeout`을 걸어두고, `raise_for_status()`로 실패를 바로 알아챌 수 있게 합니다.

한 가지 실전 팁도 함께 넣었습니다: 서버가 응답 헤더에 `charset`을 명시하지 않으면 `requests`는 인코딩을 `ISO-8859-1`로 잘못 추측해서 `£`, `–` 같은 특수문자가 `Â£`처럼 깨질 수 있습니다. `response.apparent_encoding`(콘텐츠 바이트를 보고 추정한 실제 인코딩)으로 보정해줍니다.

In [ ]:
import time
import requests

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; tutorial-crawler/1.0)"}


def fetch(url: str) -> requests.Response:
    response = requests.get(url, headers=HEADERS, timeout=10)
    response.raise_for_status()  # 200이 아니면 여기서 바로 예외를 던진다
    if response.encoding is None or response.encoding.lower() == "iso-8859-1":
        # charset 헤더가 없을 때 requests의 기본 추측(ISO-8859-1)이 틀리는 경우가 많아 보정한다
        response.encoding = response.apparent_encoding
    return response


resp = fetch("https://books.toscrape.com/")
print("status:", resp.status_code)
print("content-type:", resp.headers.get("Content-Type"))
print("본문 길이:", len(resp.text))
print(resp.text[:300])

## 실습 2. `BeautifulSoup`으로 목록 페이지 파싱하기

책 목록 페이지에서 제목과 가격만 뽑아냅니다. `.select()`는 CSS 선택자로 원하는 태그를 찾는 방법입니다.

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(resp.text, "html.parser")

books = []
for article in soup.select("article.product_pod"):
    title = article.h3.a["title"]
    price = article.select_one("p.price_color").get_text(strip=True)
    books.append({"title": title, "price": price})

print(f"{len(books)}권 파싱 완료")
for b in books[:5]:
    print(b)

## 실습 3. `crawl.py`의 `extract_text_from_html` 재구현

실제 예제 프로젝트의 `crawl.py`는 HTML에서 `<script>`/`<style>` 태그를 제거하고, 본문 글자만 줄바꿈 기준으로 깔끔하게 정리해서 뽑아냅니다. 그대로 재구현해보고 quotes.toscrape.com 페이지에 적용해봅니다.

In [ ]:
def extract_text_from_html(html: str) -> str:
    """HTML에서 사람이 읽는 본문 글자만 뽑아낸다 (crawl.py와 동일한 로직)."""
    soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)


quotes_resp = fetch("https://quotes.toscrape.com/")
clean_text = extract_text_from_html(quotes_resp.text)
print(clean_text[:500])

## 실습 4. `python-dotenv`로 설정값 관리하기

API 키나 접속 정보처럼 코드에 직접 적으면 안 되는 값은 `.env` 파일에 따로 두고 `load_dotenv()`로 불러옵니다. Colab에서는 파일을 직접 만들어 확인해봅니다.

In [ ]:
with open(".env", "w", encoding="utf-8") as f:
    f.write("DATABASE_URL=sqlite:///crawled.db\n")
    f.write("CRAWL_DELAY_SECONDS=0.5\n")

from dotenv import load_dotenv
import os

load_dotenv()

# config.py와 동일한 패턴: os.getenv(키, 기본값)
DATABASE_URL = os.getenv("DATABASE_URL", "sqlite:///default.db")
CRAWL_DELAY_SECONDS = float(os.getenv("CRAWL_DELAY_SECONDS", "1.0"))

print("DATABASE_URL:", DATABASE_URL)
print("CRAWL_DELAY_SECONDS:", CRAWL_DELAY_SECONDS)

## 실습 5. `sqlite3`로 크롤링 결과 저장하기

`db.py`의 `CREATE_TABLE_SQL`/`UPSERT_SQL` 패턴을 sqlite3 문법으로 그대로 옮깁니다. `ON CONFLICT ... DO UPDATE`(UPSERT)는 ANSI SQL 표준 자체는 아니지만, PostgreSQL과 SQLite가 공통으로 지원하는 확장 문법이라 이 노트북의 sqlite3 코드가 실제 PostgreSQL에서도 거의 그대로 동작합니다.

In [ ]:
import sqlite3

CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS crawled_documents (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    url TEXT NOT NULL UNIQUE,
    content_type TEXT NOT NULL,
    text_content TEXT,
    crawled_at TEXT NOT NULL DEFAULT (datetime('now'))
);
"""

UPSERT_SQL = """
INSERT INTO crawled_documents (url, content_type, text_content, crawled_at)
VALUES (?, ?, ?, datetime('now'))
ON CONFLICT(url) DO UPDATE SET
    content_type = excluded.content_type,
    text_content = excluded.text_content,
    crawled_at = datetime('now');
"""


def init_db(db_path: str):
    with sqlite3.connect(db_path) as conn:
        conn.execute(CREATE_TABLE_SQL)


def save_document(db_path: str, url: str, content_type: str, text_content: str):
    with sqlite3.connect(db_path) as conn:
        conn.execute(UPSERT_SQL, (url, content_type, text_content))


DB_PATH = "crawled.db"
init_db(DB_PATH)
print("테이블 준비 완료")

이제 `crawl.py`의 `main()`과 같은 방식으로 여러 URL을 순회하며 저장합니다. 요청 사이에 `CRAWL_DELAY_SECONDS`만큼 쉬어가는 것도 그대로 재현합니다.

In [ ]:
CRAWL_TARGETS = [
    "https://quotes.toscrape.com/",
    "https://quotes.toscrape.com/page/2/",
    "https://quotes.toscrape.com/tag/love/",
]

for url in CRAWL_TARGETS:
    try:
        print(f"크롤링 중: {url}")
        page = fetch(url)
        text = extract_text_from_html(page.text)
        save_document(DB_PATH, url, "html", text)
    except requests.RequestException as e:
        print(f"실패: {url} ({e})")

    time.sleep(CRAWL_DELAY_SECONDS)

print("크롤링 완료")

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    rows = conn.execute(
        "SELECT id, url, content_type, length(text_content), crawled_at FROM crawled_documents"
    ).fetchall()

for row in rows:
    print(row)

### 참고: 실제 `psycopg2` 버전

DB 서버(PostgreSQL)가 있다면 위 코드는 아래처럼 거의 그대로 옮겨갑니다 (`db.py` 원본):

```python
import psycopg2

def get_connection():
    return psycopg2.connect(DATABASE_URL)

def save_document(url, content_type, text_content, binary_content):
    with get_connection() as conn:
        with conn.cursor() as cur:
            binary_param = psycopg2.Binary(binary_content) if binary_content is not None else None
            cur.execute(UPSERT_SQL, (url, content_type, text_content, binary_param))
        conn.commit()
```
핵심 차이는 두 가지뿐입니다: 파라미터 자리표시자가 sqlite3는 `?`, psycopg2는 `%s`를 쓰고, psycopg2는 바이너리(PDF 등)를 넣을 때 `psycopg2.Binary()`로 감싸줘야 합니다.

## 실습 6. PDF처럼 바이너리 데이터는 다르게 저장하기

`crawl_one()`은 URL이 `.pdf`로 끝나면 텍스트로 바꾸지 않고 원본 바이트 그대로 저장합니다 (PDF 안의 글자를 뽑는 건 다음 단계인 청킹 노트북의 몫입니다). sqlite3의 `BLOB` 타입에 바이너리를 저장하는 것도 함께 연습해봅니다.

In [ ]:
ALTER_SQL = "ALTER TABLE crawled_documents ADD COLUMN binary_content BLOB"
with sqlite3.connect(DB_PATH) as conn:
    try:
        conn.execute(ALTER_SQL)
    except sqlite3.OperationalError:
        pass  # 컬럼이 이미 있으면 무시


def crawl_one(url: str) -> None:
    response = fetch(url)
    if url.lower().endswith(".pdf"):
        with sqlite3.connect(DB_PATH) as conn:
            conn.execute(
                "INSERT INTO crawled_documents (url, content_type, binary_content, crawled_at) "
                "VALUES (?, 'pdf', ?, datetime('now')) "
                "ON CONFLICT(url) DO UPDATE SET binary_content = excluded.binary_content",
                (url, sqlite3.Binary(response.content)),
            )
    else:
        text = extract_text_from_html(response.text)
        save_document(DB_PATH, url, "html", text)


# PDF 링크는 실제로 호출하지 않고, 분기 로직만 확인합니다.
print("'a.pdf' -> pdf 분기:", "a.pdf".lower().endswith(".pdf"))
print("'a.html' -> pdf 분기:", "a.html".lower().endswith(".pdf"))

## 정리 & 연습 문제
- **연습 1**: `CRAWL_TARGETS`를 `quotes.toscrape.com`의 태그 페이지 3~4개로 확장해서 순회하고, 각 페이지에서 명언(quote) 텍스트 개수를 세어 출력해보세요.
- **연습 2**: 같은 URL을 두 번 크롤링해도 테이블에 중복 행이 생기지 않는지(UPSERT가 잘 동작하는지) 직접 확인해보세요. `crawled_at` 값이 두 번째 실행에서 갱신되는지도 확인해보세요.

**해설/정답**: [07_web_crawling_solutions.ipynb](07_web_crawling_solutions.ipynb)

## 다음 단계
다음 노트북([08_text_chunking](../08_text_chunking/08_text_chunking.ipynb))에서는 이렇게 모아둔 원본 텍스트/PDF를 검색에 쓸 수 있도록 잘게 자르는 **청킹**을 실습합니다.